**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment3/'
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# # This 下载s Emoji 数据集 到 your Drive
# # if it doesn't 已经存在.
# %cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/数据集s/
# !bash get_数据集s.sh
# %cd /content/drive/My\ Drive/$FOLDERNAME

# Denoising Diffusion Probabilistic Models

到目前为止，我们探索的都是判别式模型，它们被训练来产生带标签的输出。这些任务包括直接的图像分类，也包括句子生成；后者仍然可以被看作分类问题，只是标签位于词表空间，并用循环机制捕捉多词标签。现在，我们将扩展工具箱，构建一个生成式模型，使其能够生成与给定训练图像集合相似的新图像。

生成式模型有很多类型，包括 Generative Adversarial Networks（GANs）、autoregressive models、normalizing flow models 和 Variational Autoencoders（VAEs），它们都能合成令人印象深刻的图像。不过在 2020 年，Ho 等人通过把 diffusion probabilistic models 与 denoising score matching 结合，提出了 Denoising Diffusion Probabilistic Models（DDPMs）。这产生了一种既容易训练、又足以生成复杂高质量图像的生成模型。下面会给出 DDPM 的高层概览。更多细节请参考课程 slides 和原始 DDPM 论文 [1]。

# Forward Process
设 $q(x_0)$ 为干净数据集图像的分布。我们把前向加噪过程定义为一条由小加噪步骤组成的 Markov chain：

$$q(x_t | x_{t-1}) \sim N(x_t; \sqrt{1-\beta_t} x_{t-1} , \beta_t I)$$

其中逐步方差 $(\beta_1, ..., \beta_T)$ 决定 noise schedule。由于 Gaussian distribution 的性质，我们可以把 $q(x_t | x_0)$ 写成闭式形式：

$$q(x_t | x_0) \sim N(x_t; \sqrt{\bar{\alpha}_t} x_0 , (1-\bar{\alpha}_t) I)$$

其中 $\alpha_t = 1-\beta_t$，且 $\bar{\alpha}_t = \prod_{s=1}^{t}\alpha_t$。如果 noise schedule $(\beta_1, ..., \beta_T)$ 设置合适，最终分布 $q(x_T)$ 会变得与纯 Gaussian noise $N(0, I)$ 难以区分。

回忆一下，从 Gaussian distribution $x \sim N(\mu, \sigma^2)$ 采样等价于计算 $\sigma * \epsilon + \mu$，其中 $\epsilon \sim N(0, 1)$。因此，只要给定 $x_{t-1}$ 或 $x_0$，从 $q(x_t | x_{t-1})$ 或 $q(x_t | x_0)$ 采样都很直接。正因如此，前向过程很简单，不需要学习。

# Reverse Process
反向过程通过多个步骤，从纯噪声 $x_T$ 重建干净图像 $x_0$。令 $p(x_{t-1} | x_t)$ 表示 $q(x_t | x_{t-1})$ 的反向步骤。
第一个关键洞见是：学习反转每一个单独的去噪步骤，比一次性反转整个前向过程更容易。换句话说，对每个 $t$ 学习 $p(x_{t-1} | x_t)$，比直接学习 $p(x_0 | x_T)$ 更容易。

不过，学习 $p(x_{t-1} | x_t)$ 仍然有挑战。尽管 $q(x_t | x_{t-1})$ 是 Gaussian，$p(x_{t-1} | x_t)$ 可能具有任意复杂形式，而且几乎肯定不是 Gaussian。对任意分布建模并采样，远比处理 Gaussian 这样的简单参数化分布困难。

第二个关键洞见是：如果前向过程中的逐步噪声 $\beta_t$ 足够小，那么反向步骤 $p(x_{t-1} | x_t)$ 也会接近 Gaussian distribution。因此，我们只需要估计它的均值和方差。实践中，把 $p(x_{t-1} | x_t)$ 的方差设置为匹配 $\beta_t$（与前向步骤相同）效果很好。于是，学习反向过程就简化为学习均值 $\mu(x_t, t; \theta)$，其中 $\theta$ 表示神经网络参数。

# Denoising Objective
生成式模型通常通过最小化数据集样本的期望负对数似然 $\mathbb{E}[-\log{p_\theta(x_0)}]$ 来优化。每个样本的 likelihood 可以写为：$p_\theta(x_0) = p(x_T)\prod_{t=1}^T p(x_{t-1} | x_t)$。由于这个目标在许多情况下不可处理，不同类别的生成式模型会优化负对数似然的 variational lower bound。

Ho 等人证明，这个目标等价于最小化如下去噪损失：

$$\mathbb{E}_{t, x_0, \epsilon}\left[ \| \epsilon - \epsilon_\theta (\sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, t ) \|^2 \right]$$

其中 $t$ 在 1 到 T 之间均匀采样，$x_0$ 是干净样本，$\epsilon$ 从标准 Gaussian $N(0, I)$ 采样，$\epsilon_\theta$ 是一个神经网络模型，训练目标是从输入 noisy sample $x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$ 中预测噪声 $\epsilon$。换句话说，$\epsilon_\theta$ 学会对输入 noisy image 去噪。注意，这等价于预测干净样本，因为根据等式 $x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$，噪声可以由 noisy image 和 clean sample 恢复。

[1] Denoising Diffusion Probabilistic Models. Jonathan Ho, Ajay Jain, Pieter Abbeel. [Link](https://arxiv.org/pdf/2006.11239)

# 本 Notebook 的内容
我们将实现并训练一个 DDPM 模型，用于根据文本 prompt 生成小尺寸 `32 x 32` emoji 图像。首先，我们会根据论文 [1] 的公式 (4) 实现前向加噪过程。然后构建一个 UNet 模型，它接收 $x_t$ 和 $t$ 作为输入（也可以附加文本 prompt 等条件），并输出与 $x_t$ 形状相同的 tensor。最后，我们会实现去噪目标，并训练 DDPM 模型。

我们使用预训练 CLIP [2] 模型的 text encoder，将输入文本编码为 512 维向量。为了加快训练，我们已经提前对训练集中的文本数据做了编码。

[2] Learning transferable visual models from natural language supervision. Radford et. al. [Link](https://github.com/openai/CLIP)

In [ ]:
!pip install git+https://github.com/openai/CLIP.git


In [ ]:
import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import torchvision.utils as tv_utils
from cs231n.emoji_dataset import EmojiDataset
from cs231n.gaussian_diffusion import GaussianDiffusion

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-10, np.abs(x) + np.abs(y))))


In [ ]:
# First, let's load 并 可视化 数据集.
# Each sample 的 数据集 is a tuple: (image, {"text_emb": <tensor>, "text": <string>})
# 我们将 使用 a pre训练ed text-encoder 到 encode text 到 an embedding vector.
# 我们已经 pre-encoded 数据集 texts 到 embeddings 用于 faster 训练.
image_size = 32
dataset = EmojiDataset(image_size)


In [ ]:
def visualize_samples(dataset, num_samples=25, grid_size=(5, 5)):
    # Randomly sample indices
    indices = random.sample(range(len(dataset)), num_samples)
    samples = [dataset[i] for i in indices]

    # Inspect one sample
    img_shape = list(samples[0][0].shape)
    emb_shape = list(samples[0][1]["text_emb"].shape)
    print(f"One sample: (image: {img_shape}, {{ \"text_emb\": {emb_shape}, \"text\": string }})")

    # Extract images 并 texts
    images = torch.stack([sample[0] for sample in samples])  # Stack images
    texts = [sample[1]["text"] for sample in samples]  # Extract text descriptions

    # Create a grid 的 images
    grid_img = tv_utils.make_grid(images, nrow=grid_size[1], padding=2)

    # Convert 到 numpy 用于 plotting
    grid_img = grid_img.permute(1, 2, 0).numpy()

    # Plot images
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid_img)
    ax.axis("off")

    # Add text annotations
    grid_w, grid_h = grid_size
    img_w, img_h = grid_img.shape[1] // grid_w, grid_img.shape[0] // grid_h

    for i, text in enumerate(texts):
        row, col = divmod(i, grid_w)
        x, y = col * img_w, row * img_h
        ax.text(x+5, y+5, text[:30], fontsize=8, color='white', bbox=dict(facecolor='black', alpha=0.5))

    plt.show()

visualize_samples(dataset)

## q_sample

现在定义前向加噪过程。阅读 `cs231n/gaussian_diffusion.py` 中的 `GaussianDiffusion` 类。公式请参考原始 DDPM 论文 [1]。实现 `q_sample` 方法，并在下面测试。你应该看到相对误差为 0。

In [ ]:
# Test GaussianDiffusion.q_sample method
sz = 2
b = 3

diffusion = GaussianDiffusion(
      model=None,
      image_size=sz,
      timesteps=1000,
      beta_schedule="sigmoid",
)

t = torch.tensor([0, 300, 999]).long()
x_start = torch.linspace(-0.9, 0.6, b*3*sz*sz).view(b, 3, sz, sz)
noise = torch.linspace(-0.7, 0.8, b*3*sz*sz).view(b, 3, sz, sz)
x_t = diffusion.q_sample(x_start, t, noise)

expected_x_t = np.array([
    [
        [[-0.9119949, -0.86840147], [-0.8248081, -0.7812148]],
        [[-0.7376214, -0.694028], [-0.65043473, -0.6068413]],
        [[-0.563248, -0.51965463], [-0.47606122, -0.43246788]],
    ],
    [
        [[-0.42800453, -0.37039882], [-0.31279305, -0.2551873]],
        [[-0.19758154, -0.1399758], [-0.08237009, -0.024764337]],
        [[0.032841414, 0.090447165], [0.14805292, 0.20565866]],
    ],
    [
        [[0.32864183, 0.37152246], [0.41440308, 0.45728368]],
        [[0.50016433, 0.5430449], [0.5859255, 0.6288062]],
        [[0.67168677, 0.7145674], [0.757448, 0.8003287]],
    ],
]).astype(np.float32)

# Should see zero 相对误差
print("x_t error: ", rel_error(x_t.numpy(), expected_x_t))

In [ ]:
# Let's 可视化 noisy images at various timesteps.
diffusion = GaussianDiffusion(
      model=None,
      image_size=image_size,
      timesteps=1000,
)

B = 10
img = dataset[770][0]  # 3 x H x W
x_start = img[None].repeat(B, 1, 1, 1)  # B x 3 x H x W
noise = torch.randn_like(x_start)  # B x 3 x H x W
t = torch.linspace(0, 1000-1, B).long()

x_start = diffusion.normalize(x_start)
x_t = diffusion.q_sample(x_start, t, noise)
x_t = diffusion.unnormalize(x_t).clamp(0, 1)
grid_img = tv_utils.make_grid(x_t, nrow=5, padding=2)
grid_img = grid_img.permute(1, 2, 0).cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid_img)
ax.axis("off")
plt.show()

扩散模型可以被训练来预测干净图像，也可以预测噪声，因为二者可以相互推导（见上面 “Denoising Objective” 部分）。实现 `predict_start_from_noise` 和 `predict_noise_from_start` 方法，并在下面测试。你应该看到小于 `1e-5` 的相对误差。

In [ ]:
# Test `predict_noise_来自_start` 并 `predict_start_来自_noise`
sz = 2
b = 3

diffusion = GaussianDiffusion(
      model=None,
      image_size=sz,
      timesteps=1000,
      beta_schedule="sigmoid",
)

t = torch.tensor([1, 300, 998]).long()
x_start = torch.linspace(-0.91, 0.67, b*3*sz*sz).view(b, 3, sz, sz)
noise = torch.linspace(-0.73, 0.81, b*3*sz*sz).view(b, 3, sz, sz)
x_t = diffusion.q_sample(x_start, t, noise)

pred_noise = diffusion.predict_noise_from_start(x_t, t, x_start)
pred_x_start = diffusion.predict_start_from_noise(x_t, t, noise)

# Should 相对误差s 约为 1e-5 or less
print("noise error: ", rel_error(pred_noise.numpy(), noise.numpy()))
print("x_start error: ", rel_error(pred_x_start.numpy(), x_start.numpy()))

## UNet 模型

现在已经定义了前向过程，接下来定义用于输入图像去噪的 UNet 模型。UNet 是一种用于 image-to-image 任务的神经网络架构，例如分割、风格迁移等。它包含一个 encoder（或 downsampling module），把输入图像转换成空间分辨率逐渐降低、特征维度逐渐增大的层级特征。decoder（或 upsampling module）随后逐步恢复空间分辨率，并镜像 encoder 结构。在每个 decoder 层中，来自对应 encoder 层的特征会被拼接进来，为细节提供直接路径。这种方式减轻了 bottleneck 层的负担，让它们专注于捕捉高层表示，而不是记忆细粒度细节。

这里使用 UNet，是因为我们的输入和输出都是尺寸对齐的图像，维度相同：`C x H x W`。UNet 中的每个 ResNet block 还会接收一个额外输入向量，称为 context，用于条件控制。我们会通过编码 diffusion timestep 和文本 prompt 来生成 context vector。

运行下面的单元，粗略查看 UNet 架构。每个红色框表示一个 ResNet block，其中包含 2 或 3 个卷积层，并保持特征图空间分辨率不变。为简洁起见，图中省略了输入到每个 ResNet block 的 context vector。每个框下方写出的形状表示该 block 之后的输出 tensor 形状。额外箭头表示 skip connections，它们让 U-Net 能在输出中保留细粒度细节。例如，形状为 `(d, h, w)` 的 `layer1_block1` 输出会与同样形状为 `(d, h, w)` 的 `layer4_block1` 输出拼接，然后传给 `layer4_block2`。因此，`layer4_block2` 会接收形状为 `(2*d, h, w)` 的输入。

In [ ]:
from IPython.display import Image
Image(f'/content/drive/My Drive/{FOLDERNAME}/unet.png')

在 `cs231n/unet.py` 中实现 `Unet.__init__` 方法，以定义 UNet 模型的上采样和下采样 block，然后在下面测试。如果实现正确，你不应该看到任何错误。调用 `Unet(dim=d, condition_dim=condition_dim, dim_mults=(2,4))` 应该能成功创建与上图架构对应的 UNet 模型。

In [ ]:
from cs231n.unet import Unet, ResnetBlock, Downsample, Upsample

dim = 2
condition_dim = 4
dim_mults = (1, 2, 4)
unet = Unet(dim=dim, condition_dim=condition_dim, dim_mults=dim_mults)


# Check 数量 层
assert len(unet.downs) == len(dim_mults), "Number of Unet downsampling blocks is wrong."
assert len(unet.ups) == len(dim_mults), "Number of Unet upsampling blocks is wrong."


# Check 层
try:
  expected_downs_dims = [2, 2, 8, 2, 2, 8, 2, 2, 8, 2, 2, 8, 4, 4, 8, 4, 4, 8]
  downs_dims = [
      d for m in unet.downs for d in [
          m[0].dim, m[0].dim_out, m[0].context_dim, m[1].dim, m[1].dim_out, m[1].context_dim,
      ]
  ]
  assert downs_dims == expected_downs_dims, "Dimensions don't match"
except Exception as e:
  raise RuntimeError("Downsampling blocks wrongly configured") from e


try:
  expected_ups_dims = [8, 4, 8, 8, 4, 8, 4, 2, 8, 4, 2, 8, 4, 2, 8, 4, 2, 8]
  ups_dims = [
      d for m in unet.ups for d in [
          m[1].dim, m[1].dim_out, m[1].context_dim, m[2].dim, m[2].dim_out, m[2].context_dim,
      ]
  ]
  assert ups_dims == expected_ups_dims, "Dimensions don't match"
except Exception as e:
  raise RuntimeError("Upsampling blocks wrongly configured") from e

# Check 数量 参数
num_params = sum(p.numel() for p in unet.parameters())
expected_num_params = 6499
assert num_params == expected_num_params, "Unet model creation is wrong!"


补全 `cs231n/unet.py` 中的 `Unet.forward` 方法，并在下面测试。暂时不用担心 `Unet.cfg_forward` 方法。你应该看到小于 `1e-6` 的相对误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

dim = 4
condition_dim = 4
dim_mults = (2, 4)
unet = Unet(dim=dim, condition_dim=condition_dim, dim_mults=dim_mults)
unet.eval()

b = 2
h = w = 4
inp_x = torch.randn(b, 3, h, w)
inp_text_emb = torch.randn(b, condition_dim)
inp_t = torch.tensor([8, 25]).long()
out = unet.forward(x=inp_x, time=inp_t,
                   model_kwargs={"text_emb": inp_text_emb}).detach().numpy()

expected_out = np.array(
      [[[[ 0.3530089,   0.49302575,  0.55152094,  0.20567769],
         [ 1.1295223,   0.05811183,  0.2143015,   0.23898259],
         [ 0.44839963,  1.0291728,   0.18519637,  0.21956968],
         [ 0.7035912,   0.716347,    0.9166326,  -0.11552593]],
        [[ 0.04585427, -0.26847702,  0.08526254,  0.2396591 ],
         [ 0.2858743,  -0.29726404,  0.02462834,  0.2350314 ],
         [ 0.3121434,   0.3962962,   0.65541625, -0.300183  ],
         [ 0.3409412,   0.35369393,  0.37445623,  0.10297439]],
        [[ 0.49440542,  0.72878885,  0.6669705,   0.56822944],
         [ 1.0084232,   0.19204213,  0.61240125,  0.575651  ],
         [ 0.6452671,   0.91168964,  0.6458106,   0.25952405],
         [ 0.7147298,   0.7707498,   0.88941365,  0.35422024]]],
       [[[ 0.3186316,  -0.17632714,  0.440247,    0.05124736],
         [ 0.6866561,   0.3019496,   0.18366373,  0.42941707],
         [-0.09776017,  0.55849403,  0.30689716,  0.12954405],
         [ 0.3667726,   0.740231,    0.43682802,  0.08275545]],
        [[-0.29714173, -0.28515863, -0.02289355,  0.05637485],
         [-0.10230345,  0.182563,    0.8107456,   0.18209176],
         [ 0.57062995,  0.07061654,  0.2864037,   0.17969263],
         [ 0.31491554,  0.5652035,   0.3810383,   0.05889529]],
        [[ 0.14559639,  0.36408228,  0.5657894,   0.24066216],
         [ 0.4431491,   0.38140664,  1.1457629,   0.40314198],
         [ 0.6673274,   0.5922724,   0.44549578,  0.5544277 ],
         [ 0.5405652,   0.88729244,  0.66340685,  0.23414826]]]])

print("forward error: ", rel_error(out, expected_out))

# p_losses

现在模型实现已经完成，接下来编写 DDPM 的去噪训练步骤。如前所述，优化去噪损失等价于最小化数据集的期望负对数似然。完成 `cs231n/gaussian_diffusion.py` 中的 `GaussianDiffusion.p_losses` 方法，并在下面测试。你应该看到小于 `1e-6` 的相对误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

dim = 4
condition_dim = 4
dim_mults = (2, 4)
unet = Unet(dim=dim, condition_dim=condition_dim, dim_mults=dim_mults)
unet.eval()

h = w = 4
b = 3
diffusion = GaussianDiffusion(
      model=unet,
      image_size=h,
      timesteps=1000,
      beta_schedule="sigmoid",
      objective="pred_x_start",
)

inp_x = torch.randn(b, 3, h, w)
inp_model_kwargs = {"text_emb": torch.randn(b, condition_dim)}
out = diffusion.p_losses(inp_x, inp_model_kwargs)
expected_out = 30.0160

print("forward error: ", rel_error(out.item(), expected_out))

## p_sample

现在还剩最后一个组成部分。DDPM 通过迭代执行反向过程来生成样本。这个反向过程的每一次迭代都涉及从 $p(x_{t-1}|x_t)$ 采样。打开 `cs231n/gaussian_diffusion.py`，按照论文公式 (6) 实现 `p_sample` 方法。该公式描述了在给定 $x_t$ 和 $x_0$ 条件下，从前向过程 posterior 中采样；其中 $x_0$ 可以由去噪模型的输出推导得到。我们已经实现了 `sample` 方法，它会迭代调用 `p_sample`，根据输入文本生成图像。

在下面测试你的 `p_sample` 实现；你应该看到小于 `1e-6` 的相对误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

dim = 4
condition_dim = 4
dim_mults = (2,)
unet = Unet(dim=dim, condition_dim=condition_dim, dim_mults=dim_mults)
unet.eval()

h = w = 4
b = 1
inp_x_t = torch.randn(b, 3, h, w)
inp_model_kwargs = {"text_emb": torch.randn(b, condition_dim)}
t = 231

# 测试 1
diffusion = GaussianDiffusion(
      model=unet,
      image_size=h,
      timesteps=1000,
      beta_schedule="sigmoid",
      objective="pred_x_start",
)
out = diffusion.p_sample(inp_x_t, t, inp_model_kwargs).detach().numpy()

expected_out = np.array(
    [[[[ 1.1249855,   0.098786,   -0.674154,    1.3266845 ],
         [-0.21832949,  0.4874724,   1.0712924,  -0.7355726 ],
         [-0.19198017,  1.0347139,   0.03363481,  0.47500697],
         [-1.0178545,  -0.52259684, -0.11972852, -0.02105119]],
        [[ 0.06438226, -1.5796007,   0.43629852, -0.67392266],
         [-1.1564192,  -1.7009112,   0.878653,   -1.2563533 ],
         [-0.45194352,  1.0274173,   0.62859684, -0.49643248],
         [-0.18070872,  0.51325583, -0.74465716, -0.5262963 ]],
        [[ 0.76330817,  1.5878401,   1.9983996,   0.6618041 ],
         [ 0.16782297,  0.5638586,  -0.8150751,   1.9507835 ],
         [ 0.7401889,   1.2696184,   1.8468435,  -1.4796975 ],
         [ 0.6636643,  -0.83933365,  0.78857577,  0.13521622]]]])
print("forward error: ", rel_error(out, expected_out))

# 测试 2
diffusion = GaussianDiffusion(
      model=unet,
      image_size=h,
      timesteps=1000,
      beta_schedule="cosine",
      objective="pred_noise",
)
out = diffusion.p_sample(inp_x_t, t, inp_model_kwargs).detach().numpy()

expected_out = np.array(
    [[[[ 1.1139543,   0.11015256, -0.7337392,   1.3292073 ],
         [-0.286727,    0.48420003,  1.0737748,  -0.764173  ],
         [-0.22848499,  1.0443672,   0.07089197,  0.50183666],
         [-1.0453997,  -0.5474187,  -0.06630269, -0.11730157]],
        [[ 0.10919069, -1.5161378,   0.38542387, -0.6693473 ],
         [-1.0730051,  -1.6837747,   0.87086374, -1.2265388 ],
         [-0.48162907,  1.0328836,   0.6336425,  -0.47346604],
         [-0.11769794,  0.5335539,  -0.71324545, -0.47889465]],
        [[ 0.8311781,   1.6457081,   1.9775101,   0.563749  ],
         [ 0.25127694,  0.57862943, -0.8148284,   1.8744429 ],
         [ 0.75546896,  1.1215659,   1.9582558,  -1.535511  ],
         [ 0.62724066, -0.8814953,   0.77542645,  0.12667023]]]])
print("forward error: ", rel_error(out, expected_out))

## 训练

现在 DDPM 训练所需的所有组件都已具备，可以在 Emoji 数据集上训练模型。这里不需要你写代码，但建议你查看 `cs231n/ddpm_trainer.py` 中的训练代码。

在 notebook 剩余部分，我们会使用 `cs231n/exp/pretrained` 文件夹中的预训练模型；它已经在这个数据集上训练了许多迭代。不过你也可以在 Colab GPU 上训练自己的模型（确保修改 `results_folder`）。注意，在 T4 GPU 上可能需要超过 12 小时才能开始看到比较合理的生成结果。

In [ ]:
from cs231n.ddpm_trainer import Trainer

dim = 48
image_size = 32
results_folder = f"/content/drive/My Drive/{FOLDERNAME}/cs231n/exp/pretrained"
condition_dim = 512

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = Unet(
    dim=dim,
    dim_mults=(1, 2, 4, 8),
    condition_dim=condition_dim,
)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

diffusion = GaussianDiffusion(
    model,
    image_size=image_size,
    timesteps=100,  # 数量 diffusion steps
    objective="pred_noise",  # "pred_x_start" or "pred_noise"
)

dataset = EmojiDataset(image_size)

trainer = Trainer(
    diffusion,
    dataset,
    device,
    train_batch_size=256,
    weight_decay=0.0,
    train_lr=1e-3,
    train_num_steps=50000,
    results_folder=results_folder,
)

# You are not required 到 训练 it yourself.
# 训练er.训练()


In [ ]:
# Instead, 我们将 load a pre训练ed 模型.
trainer.download_pretrained()
trainer.load(70000)

In [ ]:
# Helper 函数 到 get CLIP text embedding during inference time.
from cs231n.emoji_dataset import ClipEmbed
clip_embedder = ClipEmbed(device)
def get_text_emb(text):
    return trainer.ds.embed_new_text(text, clip_embedder)

# Helper 函数 到 可视化 generations.
def show_images(img):
    # img: B x T x 3 x H x W
    plt.figure(figsize=(10, 10))
    img2 = img.clamp(0, 1).permute(0, 3, 1, 4, 2).flatten(0, 1).flatten(1, 2).cpu().numpy()
    plt.imshow(img2)
    plt.axis('off')

    plt.show()

## 采样

运行下面的单元，可视化由文本 prompt 条件控制的 emoji 生成结果。你可以自由修改 prompt，探索不同生成结果。由于我们的 emoji 数据集很小，不足以训练一个完全可泛化的 text-to-image 模型。因此，对未见过 prompt 的生成结果可能较差，也可能不忠实于输入文本（对见过的样本也可能发生，但较少见）。

为了加快采样，可以使用 GPU runtime。如果切换 runtime type，请确保重新运行整个 notebook。

In [ ]:
# text = "crying face"  # seen 样本, good generations
text = "face with cowboy hat"  # seen 样本, good generations
# text = "crying face 使用 cowboy hat"  # unseen 样本, bad generations
text_emb = get_text_emb(text)
text_emb = text_emb[None].expand(5, -1).to(device)


img = trainer.diffusion_model.sample(
    batch_size=5,
    model_kwargs={"text_emb": text_emb},
    return_all_timesteps=True
)
show_images(img[:, ::20])

## Classifier Free Guidance

生成式模型通常根据 fidelity（生成样本的质量或真实感）和 diversity（样本空间的变化性或覆盖度）来评估。对于条件生成模型，fidelity 还包括生成样本是否忠实遵循输入条件。这两个指标经常互相拉扯，形成 trade-off。Ho 等人提出了一个简单技术，叫 classifier-free guidance [3]，可以显式控制这种 trade-off。

在 classifier-free guidance 中，训练条件扩散模型 $\epsilon_\theta(x_t, t, c)$ 时，会以某个概率（通常为 0.1 到 0.2）随机丢弃条件 $c$，也就是替换为 $c=\phi$。采样的每一步去噪中，预测会更新为：
$$\epsilon_\theta(x_t, t, c) \leftarrow (w+1) \epsilon_\theta(x_t, t, c) - w \epsilon_\theta(x_t, t, \phi)$$
其中 $w$ 是正标量，即 guidance scale。换句话说，我们在每个去噪步骤中做两次预测，一次有条件、一次无条件，并把它们线性组合以偏向条件生成。$w$ 是一个超参数，会根据具体模型的评估指标调优。更高的 $w$ 会让生成结果更忠实于条件，但通常会降低多样性。

[3] Classifier-Free Diffusion Guidance. Jonathan Ho, Tim Salimans. [Link](https://arxiv.org/abs/2207.12598)

在 `cs231n/unet.py` 的 `Unet.cfg_forward` 方法中实现 classifier-free guidance，并在下面测试。你应该看到小于 `1e-6` 的相对误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

dim = 4
condition_dim = 4
dim_mults = (2, 4)
unet = Unet(dim=dim, condition_dim=condition_dim, dim_mults=dim_mults)
unet.eval()

b = 2
h = w = 4
inp_x = torch.randn(b, 3, h, w)
inp_text_emb = torch.randn(b, condition_dim)
inp_t = torch.tensor([8, 25]).long()
out = unet.forward(x=inp_x, time=inp_t,
                   model_kwargs={"text_emb": inp_text_emb, "cfg_scale": 2.31}
                   ).detach().numpy()
expected_out = np.array(
      [[[[ 0.3063889,   0.49175858,  0.53406465,  0.17793494],
         [ 1.0605793,   0.07291106,  0.26333052,  0.2807781 ],
         [ 0.47007525,  1.0033083,   0.15596265,  0.21308547],
         [ 0.6635294,   0.69716704,  0.7332697,  -0.14285703]],
        [[-0.14788617, -0.17531544,  0.06930217,  0.24703366],
         [ 0.2998472,  -0.25145096,  0.06968095,  0.28924388],
         [ 0.311122,    0.35599577,  0.6346719,  -0.30859607],
         [ 0.3706389,   0.33177835,  0.29829532,  0.11159261]],
        [[ 0.34566414,  0.79019654,  0.6489645,   0.5683719 ],
         [ 0.98117805,  0.176887,    0.6220584,   0.6494926 ],
         [ 0.66813314,  0.90341353,  0.65361,     0.21999073],
         [ 0.73401093,  0.75344837,  0.71971107,  0.32417077]]],
       [[[ 0.46322435, -1.3915588,   1.0951779,   0.37392217],
         [ 0.6877824,   0.14802843, -0.7159017,   0.22476494],
         [-0.28032526,  0.50201464,  0.0972808,   0.23250426],
         [ 0.38375473,  0.86657,     0.4424622,   0.2161349 ]],
        [[-0.34920466, -0.48507357,  0.06529156,  0.29454708],
         [-0.62153226,  0.0083245,   1.7356712,  -0.0690068 ],
         [ 0.7745638,  -0.11548355,  0.237822,    0.22680345],
         [ 0.06337905,  0.7001358,   0.36441565,  0.17163567]],
        [[ 0.16040441, -0.2551211,   0.8887521,   0.11380547],
         [-0.12803352,  0.2783994,   1.4620805,   0.19044566],
         [ 0.9112923,   0.75666,     0.3765695,   0.66629255],
         [ 0.3461262,   1.0881201,   0.6392708,   0.31749305]]]])


print("forward error: ", rel_error(out, expected_out))

运行下面的单元，使用 classifier-free guidance 可视化 emoji 生成结果。你也可以自由修改 `"cfg_scale"` 参数值。如前所述，由于我们的模型泛化能力有限，即使 guidance scale 很高，也不一定能观察到忠实的生成结果。

In [ ]:
# text = "crying face"  # seen 样本, good generations
text = "face with cowboy hat"  # seen 样本, good generations
# text = "crying face 使用 cowboy hat"  # unseen 样本, bad generations
text_emb = get_text_emb(text)
text_emb = text_emb[None].expand(5, -1).to(device)


img = trainer.diffusion_model.sample(
    batch_size=5,
    model_kwargs={"text_emb": text_emb, "cfg_scale": 1},
    return_all_timesteps=True
)
show_images(img[:, ::20])